In [ ]:
# LangGraph_nodes.py
# 定义 AgentState 与所有节点工厂函数,用于构建法律问答的 LangGraph 工作流

import json
from typing import Dict, List, Any, Callable

from langchain_core.language_models import BaseLanguageModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel


# class AgentState(BaseModel):
#     query: str  # 当前用户输入
#     messages: List[Dict[str, str]]  # 对话历史 (role, content)
#     is_legal: bool  # 是否为法律问题
#     difficulty: str  # "简单" / "困难" 等
#     retrieved_docs: List[Dict]  # RAG检索结果
#     evaluation: Dict[str, List[Dict]]  # 评估分类 {"correct":[], "ambiguous":[], "incorrect":[]}
#     web_results: List[str]  # 联网搜索结果
#     crag_context: str  # 最终拼装好的上下文
#     answer: str  # 最终回答
#     pdf_path: str  # 生成的PDF路径
#     should_continue: bool  # 是否继续下一轮对话（可由外部控制）

class AgentState(BaseModel):
    # 用户问题
    query: str

    # 历史消息 分角色user agent : content
    messages: List[Dict[str, str]]

    # 是否为法律问题
    is_law_questions: bool

    # 是否为简单问题
    is_simple_questions: bool

    # RAG检索出的内容
    rag_retrieved: str

    # 评估等级 Correct Ambiguous Incorrect
    evaluation: Dict[str, List[Dict]]

    # 联网搜索结果
    web_search_results: List[str]

    # CRAG拼接检索结果
    crag_context: str

    # 上下文组装提示词送入llm
    final_prompts: str

    # 最终的回答
    final_answer: str

    # 是否需要pdf形式输出
    is_pdf_output: bool

    # 循环下一轮回答
    should_loop: bool


def start_node(state: AgentState) -> dict:
    """
    开始节点:获取用户信息,初始化必要字段。
    预期外部调用时已经将 query 放入 state。
    """
    # 将当前 query 添加进消息历史
    messages = state.get("messages", [])
    if state["query"].strip():
        messages.append({"role": "user", "content": state["query"]})
    return {
        "messages": messages,
        "is_legal": False,
        "difficulty": "",
        "retrieved_docs": [],
        "evaluation": {"correct": [], "ambiguous": [], "incorrect": []},
        "web_results": [],
        "crag_context": "",
        "answer": "",
        "pdf_path": "",
        "should_continue": True
    }


def create_router_node(llm: BaseLanguageModel) -> Callable:
    """
    返回路由节点函数:使用少样本LLM判断问题法律相关性与难度。
    """
    ROUTER_PROMPT = PromptTemplate.from_template("""
    你是一个法律咨询系统的智能路由器。请分析用户提问,完成两项判断:
    1. 该问题是否与法律相关（回答“是”或“否”）；
    2. 如果相关,判断其复杂度（“简单”或“困难”）。
       - 简单:仅需常识性法律知识即可回答,无需查阅具体法条或案例。
       - 困难:需要检索法律条文,案例或进行深度推理。
    请严格输出JSON:{{"is_legal": true/false, "difficulty": "简单/困难/不适用"}}
    
    示例:
    用户:什么是合同？ -> {{"is_legal": true, "difficulty": "简单"}}
    用户:今天天气如何？ -> {{"is_legal": false, "difficulty": "不适用"}}
    用户:婚姻中隐匿财产离婚如何分割？ -> {{"is_legal": true, "difficulty": "困难"}}
    
    现在请判断:
    {query}
        """)
    chain = ROUTER_PROMPT | llm | StrOutputParser()

    def router_node(state: AgentState) -> dict:
        query = state["query"]
        raw = chain.invoke({"query": query})
        try:
            result = json.loads(raw)
            is_legal = result.get("is_legal", False)
            difficulty = result.get("difficulty", "不适用") if is_legal else "不适用"
        except Exception:
            is_legal = False
            difficulty = "不适用"
        return {"is_legal": is_legal, "difficulty": difficulty}

    return router_node


def simple_llm_node(state: AgentState, llm: BaseLanguageModel) -> dict:
    """
    简单问题直接回答节点。
    """
    SIMPLE_PROMPT = PromptTemplate.from_template(
        """
        你是一个智能,可靠,友善的AI助手。请遵循以下原则

        1. 准确优先:不确定时不编造,明确说“我不知道”或“需要进一步信息”
        2. 简洁清晰:直接回答问题核心,避免啰嗦和无关内容
        3. 安全中立:不生成暴力,色情,仇恨或歧视性内容；不提供违法建议
        4. 结构友好:当需要分点,列表或步骤时,使用清晰的结构（如序号或短横）
        5. 仅输出回答内容:不要添加“作为AI模型...”之类的开场白或免责声明,除非用户明确询问你的身份
        """
    )
    chain = SIMPLE_PROMPT | llm | StrOutputParser()

    answer = chain.invoke({"query": state["query"]})
    messages = state.get("messages", [])
    messages.append({"role": "assistant", "content": answer})
    return {"answer": answer, "messages": messages}


def create_retrieval_node(rag_service: Any) -> Callable:
    """
    返回RAG检索节点:调用混合检索,返回文档列表。
    rag_service 需实现 search_with_rerank(query, namespace, top_k) 方法
    :param rag_service:
    :return:
    """

    def retrieval_node(state: AgentState) -> dict:
        # 假设 rag_service 有 search_with_rerank 方法
        docs = rag_service.search_with_rerank(
            query=state["query"],
            namespace="law_cases",
            top_k=20,
            rerank_top_n=10
        )
        # 将检索结果转为字典列表
        retrieved = []
        for doc in docs:
            retrieved.append({
                "id": doc.id,
                "chunk_text": doc.metadata.get("chunk_text", ""),
                "rerank_score": getattr(doc, "rerank_score", None),
                "metadata": doc.metadata
            })
        return {"retrieved_docs": retrieved}

    return retrieval_node


def create_evaluate_node(llm: BaseLanguageModel, correct_threshold: float = 0.7,
                         incorrect_threshold: float = 0.3) -> Callable:
    """
    返回评估节点:基于检索文档的重排分数或LLM判断,将文档分为 correct/ambiguous/incorrect。
    这里使用重排分数简单演示（假设 retrieved_docs 包含 rerank_score）。
    """

    def evaluate_node(state: AgentState) -> dict:
        docs = state.get("retrieved_docs", [])
        correct, ambiguous, incorrect = [], [], []
        for doc in docs:
            score = doc.get("rerank_score", 0.0)
            if score >= correct_threshold:
                correct.append(doc)
            elif score < incorrect_threshold:
                incorrect.append(doc)
            else:
                ambiguous.append(doc)
        return {"evaluation": {"correct": correct, "ambiguous": ambiguous, "incorrect": incorrect}}

    return evaluate_node


def create_web_search_node(search_func: Callable[[str, int], List[str]]) -> Callable:
    """
    返回联网检索节点:对 ambiguous 和 incorrect 的内容进行外部搜索。
    search_func(query, num) -> List[str] 返回片段列表。
    """

    def web_search_node(state: AgentState) -> dict:
        evaluation = state.get("evaluation", {})
        # 混合需要补充知识的文档文本,构建搜索查询
        docs_to_search = evaluation.get("ambiguous", []) + evaluation.get("incorrect", [])
        if not docs_to_search:
            return {"web_results": []}
        # 简单拼接文档前200字符作为二次搜索关键词,或用原query
        search_query = state["query"]  # 直接用用户问题
        results = search_func(search_query, num=5)
        return {"web_results": results}

    return web_search_node


def create_analysis_node(llm: BaseLanguageModel) -> Callable:
    """
    LLM分析节点:组装 correct + ambiguous + web_results 为上下文,生成最终答案。
    """

    def analysis_node(state: AgentState) -> dict:
        evaluation = state.get("evaluation", {})
        correct = evaluation.get("correct", [])
        ambiguous = evaluation.get("ambiguous", [])
        web_results = state.get("web_results", [])

        # 构建上下文
        context_parts = []
        for doc in correct:
            context_parts.append(f"[高相关案例] {doc['chunk_text']}")
        for doc in ambiguous:
            context_parts.append(f"[中等相关案例] {doc['chunk_text']}")
        for i, snippet in enumerate(web_results, 1):
            context_parts.append(f"[外部资料{i}] {snippet}")
        context = "\n\n".join(context_parts)

        prompt = PromptTemplate.from_template(
            "你是资深法律顾问。请根据以下资料回答用户问题。\n"
            "资料:\n{context}\n\n"
            "用户问题:{query}\n"
            "详细分析并给出法律建议:"
        )
        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({"context": context, "query": state["query"]})

        messages = state.get("messages", [])
        messages.append({"role": "assistant", "content": answer})
        return {
            "crag_context": context,
            "answer": answer,
            "messages": messages
        }

    return analysis_node


def create_postprocess_node(pdf_generator: Callable[[str, str], str]) -> Callable:
    """
    后处理节点:将最终回答保存为PDF,并更新pdf_path。
    pdf_generator(answer_text, output_filename) -> pdf_path
    """

    def postprocess_node(state: AgentState) -> dict:
        answer = state.get("answer", "")
        pdf_path = pdf_generator(answer, f"report_{state['query'][:20]}.pdf")
        return {"pdf_path": pdf_path}

    return postprocess_node


# 可选:记忆更新 / 循环控制节点
def memory_update_node(state: AgentState) -> dict:
    """
    保留记忆并准备下一轮。将 should_continue 置为 True 可继续循环。
    """
    # 简单重置部分字段,保留messages历史
    return {
        "query": "",  # 清空查询,等待下一轮输入
        "should_continue": True,
        "is_legal": False,
        "difficulty": "",
        "retrieved_docs": [],
        "evaluation": {"correct": [], "ambiguous": [], "incorrect": []},
        "web_results": [],
        "crag_context": "",
        "answer": "",
        "pdf_path": ""
    }


def non_legal_node(state: AgentState) -> dict:
    """
    非法律问题直接拒答。
    """
    answer = "抱歉,我只能回答法律相关的问题,请提出法律方面的疑问。"
    messages = state.get("messages", [])
    messages.append({"role": "assistant", "content": answer})
    return {"answer": answer, "messages": messages, "should_continue": False}


# 条件路由函数（必须单独定义,因为条件边需要引用函数名称,此处提供样例）
def route_by_difficulty(state: AgentState) -> str:
    if not state["is_legal"]:
        return "non_legal"
    if state["difficulty"] == "简单":
        return "simple_llm"
    else:
        return "crag_retrieval"  # 困难或其它情况进入CRAG检索


def route_after_evaluation(state: AgentState) -> str:
    eval = state.get("evaluation", {})
    correct_count = len(eval.get("correct", []))
    ambiguous_count = len(eval.get("ambiguous", []))
    incorrect_count = len(eval.get("incorrect", []))
    # 如果正确文档足够,直接分析；否则需要联网搜索
    if correct_count >= 3:
        return "analysis"
    else:
        return "web_search"


# 提供给外部使用的主节点字典和路由函数
NODES = {
    "start": start_node,
    "router": create_router_node,
    "simple_llm": simple_llm_node,
    "non_legal": non_legal_node,
    "retrieval": create_retrieval_node,
    "evaluate": create_evaluate_node,
    "web_search": create_web_search_node,
    "analysis": create_analysis_node,
    "postprocess": create_postprocess_node,
    "memory_update": memory_update_node
}

